# CI/CD Pipelines for Machine Learning

## Learning Objectives
- Understand CI/CD concepts for ML projects
- Create GitHub Actions workflows
- Automate testing and model validation
- Set up deployment pipelines

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

# Create directory for CI/CD examples
cicd_dir = Path("../cicd-examples")
cicd_dir.mkdir(exist_ok=True)
(cicd_dir / ".github" / "workflows").mkdir(parents=True, exist_ok=True)

print("CI/CD examples directory created!")

## 1. CI/CD for ML: Why It's Different

ML CI/CD pipelines must handle:

- **Code testing**: Unit tests, integration tests
- **Data validation**: Schema checks, quality tests
- **Model validation**: Performance thresholds
- **Model retraining**: Automated when data changes

In [ ]:
# Basic GitHub Actions workflow for ML
basic_workflow = '''name: ML Pipeline

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    
    steps:
    - uses: actions/checkout@v4
    
    - name: Set up Python
      uses: actions/setup-python@v5
      with:
        python-version: "3.12"
    
    - name: Install dependencies
      run: |
        pip install --upgrade pip
        pip install -r requirements.txt
        pip install pytest pytest-cov
    
    - name: Run tests
      run: |
        pytest tests/ -v --cov=src --cov-report=xml
    
    - name: Upload coverage
      uses: codecov/codecov-action@v3
      with:
        file: ./coverage.xml
'''

workflow_path = cicd_dir / ".github" / "workflows" / "ml-pipeline.yml"
workflow_path.write_text(basic_workflow)

print("Basic ML Pipeline Workflow:")
print(basic_workflow)

## 2. Model Training Pipeline

In [ ]:
# Model training workflow
training_workflow = '''name: Model Training

on:
  workflow_dispatch:  # Manual trigger
    inputs:
      model_name:
        description: "Model name"
        required: true
        default: "classifier"
  schedule:
    - cron: "0 0 * * 0"  # Weekly on Sunday

jobs:
  train:
    runs-on: ubuntu-latest
    
    steps:
    - uses: actions/checkout@v4
    
    - name: Set up Python
      uses: actions/setup-python@v5
      with:
        python-version: "3.12"
    
    - name: Install dependencies
      run: pip install -r requirements.txt
    
    - name: Train model
      run: python scripts/train.py --model ${{ github.event.inputs.model_name }}
    
    - name: Validate model
      run: python scripts/validate.py --min-accuracy 0.85
    
    - name: Upload model artifact
      uses: actions/upload-artifact@v4
      with:
        name: trained-model
        path: models/
'''

train_path = cicd_dir / ".github" / "workflows" / "train.yml"
train_path.write_text(training_workflow)

print("Model Training Workflow:")
print(training_workflow)

## 3. Model Validation Script

In [ ]:
# Create validation script
validation_script = '''#!/usr/bin/env python3
"""Model validation script for CI/CD."""

import argparse
import sys
import json
from pathlib import Path

def validate_model(model_path: str, min_accuracy: float = 0.85):
    """Validate model meets performance threshold."""
    
    # Load metrics
    metrics_path = Path(model_path) / "metrics.json"
    if not metrics_path.exists():
        print("ERROR: No metrics file found")
        return False
    
    with open(metrics_path) as f:
        metrics = json.load(f)
    
    accuracy = metrics.get("accuracy", 0)
    
    print(f"Model accuracy: {accuracy:.4f}")
    print(f"Minimum required: {min_accuracy:.4f}")
    
    if accuracy >= min_accuracy:
        print("PASS: Model meets accuracy threshold")
        return True
    else:
        print("FAIL: Model below accuracy threshold")
        return False


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-path", default="models/")
    parser.add_argument("--min-accuracy", type=float, default=0.85)
    args = parser.parse_args()
    
    success = validate_model(args.model_path, args.min_accuracy)
    sys.exit(0 if success else 1)
'''

scripts_dir = cicd_dir / "scripts"
scripts_dir.mkdir(exist_ok=True)
(scripts_dir / "validate.py").write_text(validation_script)

print("Validation script created!")
print(validation_script)

## 4. Deployment Pipeline

In [ ]:
# Deployment workflow
deploy_workflow = '''name: Deploy Model

on:
  release:
    types: [published]

jobs:
  deploy:
    runs-on: ubuntu-latest
    environment: production
    
    steps:
    - uses: actions/checkout@v4
    
    - name: Download model artifact
      uses: actions/download-artifact@v4
      with:
        name: trained-model
        path: models/
    
    - name: Build Docker image
      run: |
        docker build -t ml-api:${{ github.ref_name }} .
    
    - name: Push to registry
      run: |
        echo ${{ secrets.DOCKER_PASSWORD }} | docker login -u ${{ secrets.DOCKER_USERNAME }} --password-stdin
        docker push ml-api:${{ github.ref_name }}
    
    - name: Deploy to production
      run: |
        echo "Deploying version ${{ github.ref_name }}"
        # kubectl apply -f k8s/deployment.yaml
'''

deploy_path = cicd_dir / ".github" / "workflows" / "deploy.yml"
deploy_path.write_text(deploy_workflow)

print("Deployment Workflow:")
print(deploy_workflow)

## 5. Pre-commit Hooks for ML

In [ ]:
# Pre-commit configuration
precommit_config = '''repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.5.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-json
      - id: check-added-large-files
        args: ["--maxkb=500"]

  - repo: https://github.com/psf/black
    rev: 24.1.0
    hooks:
      - id: black

  - repo: https://github.com/pycqa/isort
    rev: 5.13.2
    hooks:
      - id: isort

  - repo: https://github.com/pycqa/flake8
    rev: 7.0.0
    hooks:
      - id: flake8

  - repo: local
    hooks:
      - id: pytest
        name: pytest
        entry: pytest tests/ -v --tb=short
        language: system
        pass_filenames: false
        always_run: true
'''

(cicd_dir / ".pre-commit-config.yaml").write_text(precommit_config)

print("Pre-commit configuration:")
print(precommit_config)

## Summary

### CI/CD Best Practices for ML

1. **Continuous Integration**: Run tests on every commit
2. **Model Validation**: Enforce performance thresholds
3. **Automated Training**: Schedule or trigger retraining
4. **Artifact Management**: Store models and metrics
5. **Staged Deployment**: Dev → Staging → Production

In [ ]:
print("=" * 50)
print("CI/CD PIPELINES COMPLETE")
print("=" * 50)
print("\nFiles Created:")
for f in cicd_dir.rglob("*"):
    if f.is_file():
        print(f"  📄 {f.relative_to(cicd_dir)}")